# Paso 5: consolidar el corpus ampliado v2.0

Junta en **un solo manifiesto** todo lo que hay hasta ahora y lo empaqueta en `dataset_v2.0.zip`:

| tipo | de dónde sale | split |
|---|---|---|
| `real` | `dataset_v1.0.zip` + `split_v1.csv` | train / dev / test_sk001 / test_otros (la partición de siempre) |
| `real` (control) | `no_pareado_v1.zip` → `metadata_control.csv` (transcripción humana) | **test_control** |
| `tts` | `audio_generado/metadata_tts.csv` (salida de `filtrar_sintetico`) | train |
| `aug` | `aug_v1.zip` → `metadata_aug.csv` | train |
| `pseudo` | `pseudo_v1/metadata_pseudo.csv` + audio de `no_pareado_v1.zip` | train |

Todo el audio queda a 16 kHz, mono, PCM 16-bit, en `wavs/<tipo>/`. El entrenamiento elige qué tipos usar
filtrando la columna `tipo`; así cada experimento (real, +aug, +tts, +pseudo) sale del mismo corpus.

**Chequeos antes de cerrar** (si uno falla, el notebook se detiene y no escribe nada):
1. ninguna frase de dev/test aparece en el texto del TTS (ni contenida en una frase más larga);
2. el aumento sale solo de ids de train;
3. el audio no pareado no viene de las mismas fuentes (instituciones / canales) que test_otros;
4. las pseudoetiquetas no incluyen clips del control;
5. ids únicos y todos los archivos existen.

**Sale:** `dataset_v2.0.zip` + su `.sha256`, y la tabla de horas por tipo (`stats.md`) para la tesis.
No necesita GPU.

In [ ]:
#@title 1. Rutas (solo cambia BASE si tu carpeta TESIS está en otro lugar)
BASE = "/content/drive/MyDrive/TESISOTA/TESIS"  #@param {type:"string"}
INCLUIR_PSEUDO = True  #@param {type:"boolean"}
VERSION = "2.0"
SR = 16000

DATASET_ZIP = f"{BASE}/asr/dataset_v1.0.zip"
SPLIT_CSV = f"{BASE}/asr/split_v1.csv"
AUG_ZIP = f"{BASE}/asr/aug_v1.zip"
NP_ZIP = f"{BASE}/asr/no_pareado_v1.zip"
AUDIO_GEN = f"{BASE}/audio_generado"
PSEUDO_CSV = f"{BASE}/asr/pseudo_v1/metadata_pseudo.csv"
SALIDA = f"{BASE}/asr"

## 2. Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Abrir las fuentes

In [ ]:
import hashlib, json, re, shutil, unicodedata, zipfile
from concurrent.futures import ThreadPoolExecutor
from math import gcd
from pathlib import Path
import numpy as np, pandas as pd, soundfile as sf
from scipy.signal import resample_poly


def sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def normalizar(t):   # = text_norm de build_dataset_v1.py
    t = unicodedata.normalize("NFC", str(t)).lower()
    t = re.sub(r"[¡!¿?.,;:…\"“”«»()\[\]{}—–\-_/\\]", " ", t)
    t = t.replace("'", " ").replace("’", " ")
    return re.sub(r"\s+", " ", t).strip()


def abrir(zip_path):
    z = Path(zip_path)
    if not z.is_file():
        raise SystemExit(f"Falta {z}")
    destino = Path("/content/fuentes") / z.stem
    if not destino.exists():
        tmp = destino.with_name(destino.name + ".tmp"); shutil.rmtree(tmp, ignore_errors=True)
        with zipfile.ZipFile(z) as f:
            f.extractall(tmp)
        tmp.rename(destino)
    return destino


faltan = [p for p in (DATASET_ZIP, SPLIT_CSV, AUG_ZIP, NP_ZIP, f"{AUDIO_GEN}/metadata_tts.csv")
          + ((PSEUDO_CSV,) if INCLUIR_PSEUDO else ()) if not Path(p).exists()]
if faltan:
    raise SystemExit("Faltan en Drive:\n  " + "\n  ".join(faltan))

R_REAL = next(abrir(DATASET_ZIP).rglob("metadata.csv")).parent
R_AUG = next(abrir(AUG_ZIP).rglob("metadata_aug.csv")).parent
R_NP = next(abrir(NP_ZIP).rglob("metadata_control.csv")).parent
R_TTS = Path(AUDIO_GEN)
fuentes_sha = {Path(p).name: sha256(p) for p in (DATASET_ZIP, AUG_ZIP, NP_ZIP)}
fuentes_sha["split_v1.csv"] = sha256(SPLIT_CSV)
fuentes_sha["metadata_tts.csv"] = sha256(R_TTS / "metadata_tts.csv")
if INCLUIR_PSEUDO:
    fuentes_sha["metadata_pseudo.csv"] = sha256(PSEUDO_CSV)
for k, v in fuentes_sha.items():
    print(f"{k:22s} {v[:16]}")

## 4. Manifiesto unificado

In [ ]:
COLS = ["id", "file_name", "tipo", "split", "source", "subsource", "speaker_group", "transcription", "text_norm",
        "origen_id", "avg_logprob", "ruta_fuente"]

real = pd.read_csv(R_REAL / "metadata.csv").merge(pd.read_csv(SPLIT_CSV)[["id", "split"]], on="id", how="left")
assert real.split.notna().all(), "hay clips reales sin split"
real = real.assign(tipo="real", origen_id="", avg_logprob=np.nan, ruta_fuente=[str(R_REAL / f) for f in real.file_name])
real["split"] = real.split.replace({"fuera_tts": "fuera", "fuera_texto": "fuera"})

ctrl = pd.read_csv(R_NP / "metadata_control.csv", keep_default_na=False)
ctrl = ctrl.assign(tipo="real", split="test_control", speaker_group="NP-" + ctrl.subsource, origen_id="",
                   avg_logprob=np.nan, ruta_fuente=[str(R_NP / f) for f in ctrl.file_name])

tts = pd.read_csv(R_TTS / "metadata_tts.csv", keep_default_na=False)
tts = tts.assign(tipo="tts", split="train", origen_id="", avg_logprob=np.nan,
                 ruta_fuente=[str(R_TTS / f) for f in tts.file_name])

aug = pd.read_csv(R_AUG / "metadata_aug.csv", keep_default_na=False)
aug = aug.assign(tipo="aug", split="train", avg_logprob=np.nan, ruta_fuente=[str(R_AUG / f) for f in aug.file_name])

partes = [real, ctrl, tts, aug]
if INCLUIR_PSEUDO:
    ps = pd.read_csv(PSEUDO_CSV, keep_default_na=False)
    ps = ps.assign(tipo="pseudo", split="train", origen_id="", ruta_fuente=[str(R_NP / f) for f in ps.file_name])
    partes.append(ps)

M = pd.concat([p.reindex(columns=COLS) for p in partes], ignore_index=True)
M["id"] = M.tipo.str.upper() + "-" + M.id.astype(str).str.replace(r"^(TTS-)", "", regex=True)
M["file_name"] = "wavs/" + M.tipo + "/" + M.id + ".wav"
M["text_norm"] = M.text_norm.fillna("").astype(str)
M["origen_id"] = M.origen_id.fillna("")
print(M.groupby(["tipo", "split"]).size().unstack(fill_value=0).to_string())

## 5. Chequeos

In [ ]:
ORG_YT = {"cuento1": "Educación Intercultural Perú", "cuento2": "Educación Intercultural Perú",
          "cuento3": "JOIAMAZONIA", "cuento4": "JOIAMAZONIA", "cuento5": "JOIAMAZONIA", "cuento6": "JOIAMAZONIA",
          "cuento7": "Jorge Escobar (YouTube)", "funciones_congreso": "Congreso de la República"}
ORG_NP = {"mincul_covid19": "Ministerio de Cultura", "pnud_peru_vota_seguro": "PNUD Perú",
          "pnud_respira_amazonia": "PNUD Perú", "midis_cuna_mas": "MIDIS Cuna Más",
          "grn_palabras_de_vida": "Global Recordings Network"}


def org(fila):
    if fila.source == "youtube":
        return ORG_YT[fila.subsource]
    if fila.source == "workshop":
        return "Workshop PUCP (grabación propia)"
    if fila.source == "menendez_gomez_2025":
        return "Menéndez & Gómez (2025)"
    return ORG_NP.get(fila.source, fila.source)


M["organizacion"] = [org(r) for r in M.itertuples()]
res = []

# 1) frases de dev/test dentro del texto del TTS (contenidas, no solo iguales)
blob = "\n".join(f" {t} " for t in M.loc[M.tipo == "tts", "text_norm"])
for s in ("dev", "test_sk001", "test_otros"):   # garantizado por split_v1: si falla, algo está mal
    textos = set(M.loc[(M.tipo == "real") & (M.split == s), "text_norm"])
    n = sum(f" {t} " in blob for t in textos if t)
    res.append((f"1. frases de {s} dentro del TTS", n, n == 0))
# el control no existía cuando se hizo el split: si alguna frase suya está en el TTS, ese clip sale del test
fuga_c = (M.split == "test_control") & M.text_norm.map(lambda t: bool(t) and f" {t} " in blob)
M.loc[fuga_c, "split"] = "fuera"
res.append(("1. frases de test_control dentro del TTS (se sacan del test)", int(fuga_c.sum()), True))

# 2) el aumento sale solo de train
split_de = M[M.tipo == "real"].assign(oid=lambda x: x.id.str[5:]).set_index("oid").split
malos = M.loc[M.tipo == "aug", "origen_id"].map(split_de)
res.append(("2. clips aug cuyo origen no es train", int((malos != "train").sum()), bool((malos == "train").all())))

# 3) no pareado (pseudo y control) vs fuentes de test_otros
org_test = set(M.loc[M.split == "test_otros", "organizacion"])
org_np = set(M.loc[(M.tipo == "pseudo") | (M.split == "test_control"), "organizacion"])
comun = org_test & org_np
res.append((f"3. fuentes compartidas no pareado / test_otros {sorted(comun)}", len(comun), not comun))

# 4) pseudo sin clips del control
ids_ctrl = set(M.loc[M.split == "test_control", "id"].str[5:])
n4 = int(M.loc[M.tipo == "pseudo", "id"].str[7:].isin(ids_ctrl).sum())
res.append(("4. pseudoetiquetas que son clips del control", n4, n4 == 0))

# 5) integridad
dup = int(M.id.duplicated().sum())
falt = [p for p in M.ruta_fuente if not Path(p).exists()]
res.append(("5a. ids repetidos", dup, dup == 0))
res.append(("5b. archivos que no existen", len(falt), not falt))
vacios = int((M.text_norm == "").sum())
res.append(("5c. clips sin texto", vacios, vacios == 0))

CHEQUEOS = pd.DataFrame(res, columns=["chequeo", "casos", "ok"])
print(CHEQUEOS.to_string(index=False))
if not CHEQUEOS.ok.all():
    raise SystemExit("Hay chequeos que fallan: no se escribe el corpus. Revisa la tabla.")
print("\nTodos los chequeos pasan.")

## 6. Escribir el audio (16 kHz, mono) — lee en paralelo

In [ ]:
OUT = Path("/content") / f"dataset_v{VERSION}"
for t in M.tipo.unique():
    (OUT / "wavs" / t).mkdir(parents=True, exist_ok=True)


def convertir(par):
    src, dst = par
    d = OUT / dst
    if d.exists():                                  # reanudable dentro de la sesión
        return sf.info(str(d)).frames / SR
    y, sr = sf.read(src, dtype="float32", always_2d=True)
    y = y.mean(axis=1)
    if sr != SR:
        g = gcd(sr, SR)
        y = resample_poly(y, SR // g, sr // g).astype("float32")
    sf.write(str(d), np.clip(y, -1, 1), SR, subtype="PCM_16")
    return len(y) / SR


from tqdm.auto import tqdm

# El TTS se lee desde Drive, que entrega ~1-2 archivos/s: es la parte lenta. Para no perderla si Colab se
# desconecta, se convierte por bloques y cada bloque terminado se guarda en Drive como un zip. Al volver a
# ejecutar, los bloques ya hechos se recuperan de ahí (pocos archivos grandes: segundos) y solo se lee lo que falta.
CACHE = Path(SALIDA) / f"_cache_v{VERSION}"
CACHE.mkdir(exist_ok=True)
BLOQUE = 1500
tts_idx = M.index[M.tipo == "tts"]
bloques = [tts_idx[i:i + BLOQUE] for i in range(0, len(tts_idx), BLOQUE)]
for k, b in enumerate(bloques):
    zb = CACHE / f"tts16k_{k:03d}_{len(b)}.zip"
    if zb.exists():
        with zipfile.ZipFile(zb) as z:
            z.extractall(OUT)
        print(f"bloque TTS {k + 1}/{len(bloques)}: recuperado de Drive")
        continue
    with ThreadPoolExecutor(32) as ex:      # más hilos: el límite es la latencia de Drive, no la CPU
        list(tqdm(ex.map(convertir, zip(M.ruta_fuente[b], M.file_name[b])), total=len(b),
                  desc=f"TTS bloque {k + 1}/{len(bloques)}"))
    tmp = zb.with_suffix(".tmp")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_STORED) as z:
        for f in M.file_name[b]:
            z.write(OUT / f, f)
    tmp.rename(zb)                           # solo existe si se escribió completo

# lo demás está en /content (zips descomprimidos): rápido. Lo ya convertido se salta.
with ThreadPoolExecutor(16) as ex:
    M["duration_s"] = list(tqdm(ex.map(convertir, zip(M.ruta_fuente, M.file_name)), total=len(M), desc="audio"))
M["duration_s"] = M.duration_s.round(3)

## 7. Manifiesto, tabla para la tesis y zip

In [ ]:
M.drop(columns=["ruta_fuente"]).to_csv(OUT / "metadata.csv", index=False, encoding="utf-8")
M[M.split == "train"][["file_name", "text_norm", "tipo"]].rename(columns={"text_norm": "transcription"}).to_csv(
    OUT / "metadata_hf_train.csv", index=False, encoding="utf-8")
CHEQUEOS.to_csv(OUT / "chequeos.csv", index=False)

h = M.pivot_table(index="tipo", columns="split", values="duration_s", aggfunc="sum", fill_value=0) / 3600
h = h.reindex(["real", "tts", "aug", "pseudo"]).dropna(how="all")
cols = [c for c in ["train", "dev", "test_sk001", "test_otros", "test_control", "fuera"] if c in h.columns]
h = h[cols]
h["total"] = h.sum(axis=1)
h.loc["TOTAL"] = h.sum()
clips = M.groupby("tipo").size().reindex(h.index[:-1]).tolist() + [len(M)]
tabla = h.round(2)
tabla.insert(0, "clips", clips)
print(tabla.to_string())

lineas = [f"# Corpus ampliado Shipibo-Konibo v{VERSION} — horas por tipo y partición", "",
          "| tipo | clips | " + " | ".join(tabla.columns[1:]) + " |", "|---|" + "---|" * len(tabla.columns)]
for t, r in tabla.iterrows():
    nombre = f"**{t}**" if t == "TOTAL" else t
    lineas.append(f"| {nombre} | {int(r.clips):,} | " + " | ".join(f"{r[c]:.2f}" for c in tabla.columns[1:]) + " |")
lineas += ["", "real = audio humano pareado (SK001, workshop, YouTube) y el control con transcripción humana (test_control); "
           "tts = sintético Tacotron2+HiFi-GAN filtrado; aug = velocidad ×0.9/×1.1 y ruido 10–20 dB sobre train real; "
           "pseudo = no pareado etiquetado por el ASR semilla.", "",
           "Chequeos: " + "; ".join(f"{c} = {n}" for c, n, _ in res)]
(OUT / "stats.md").write_text("\n".join(lineas), encoding="utf-8")
stats = dict(version=VERSION, sample_rate=SR, fuentes_sha256=fuentes_sha, incluye_pseudo=INCLUIR_PSEUDO,
             horas=tabla.drop(columns="clips").to_dict(), clips_por_tipo=dict(zip(tabla.index, clips)))
(OUT / "stats.json").write_text(json.dumps(stats, indent=2, ensure_ascii=False), encoding="utf-8")

ZIP_LOCAL = Path("/content") / f"dataset_v{VERSION}.zip"
with zipfile.ZipFile(ZIP_LOCAL, "w", zipfile.ZIP_STORED) as z:   # wav no se comprime: STORED es igual de chico
    for f in tqdm(sorted(OUT.rglob("*")), desc="zip"):
        if f.is_file():
            z.write(f, f.relative_to(OUT.parent))
sha = sha256(ZIP_LOCAL)
destino = Path(SALIDA) / ZIP_LOCAL.name
shutil.copy(ZIP_LOCAL, destino)
(Path(SALIDA) / f"{ZIP_LOCAL.name}.sha256").write_text(f"{sha}  {ZIP_LOCAL.name}\n")
shutil.copy(OUT / "stats.md", Path(SALIDA) / f"stats_v{VERSION}.md")
print(f"\n{destino} ({destino.stat().st_size / 1e9:.2f} GB)\nsha256 {sha}")